In [0]:
%run ../fw

In [0]:
from pyspark.sql import DataFrame

In [0]:
VOLUME = "/Volumes/workspace/e-commerce/e-commerce_sales_analysis"
CHECKPOINT = f"{VOLUME}/_pipeline/checkpoints/silver_from_sink"

silver_layer = SilverLayer(
    bronze_table_name="workspace.`e-commerce`.sink",
    silver_table_name="workspace.`e-commerce`.silver_orders",
    bad_record_table_name="workspace.`e-commerce`.silver_sales_bad_record",
    schema_detail={
        "order_id": "BIGINT",
        "country": "STRING",
        "category": "STRING",
        "unit_price": "DECIMAL(18,2)",
        "quantity": "INT",
        "order_date": "DATE",
        "total_amount": "DECIMAL(18,2)",
    },
    keys=["order_id"],

)


In [0]:
def process_microbatch(batch_df: DataFrame, batch_id: int) -> None:
    bronze_df = silver_layer.read_add_sk_from_bronze_table(batch_df)
    invalid_df = silver_layer.get_invalid_record(bronze_df)
    key_null_df = silver_layer.get_key_null_record(bronze_df)
    valid_df = (
        bronze_df
        .join(invalid_df.select("_sk"), ["_sk"], "left_anti")
        .join(key_null_df.select("_sk"), ["_sk"], "left_anti")
    )
    dup_df = silver_layer.get_dup_record(valid_df, key_null_df)
    all_bad_df = silver_layer.get_all_bad_record(invalid_df, key_null_df, dup_df)
    final_result_df = silver_layer.get_final_result(valid_df, all_bad_df)

    silver_layer.load_bad_record(all_bad_df)
    silver_layer.load_to_silver_layer(final_result_df)
    print(f"Processed Silver micro-batch {batch_id}")


query = (
    spark.readStream
    .option("schemaTrackingLocation", f"{CHECKPOINT}/schema_tracking")
    .option("allowSourceColumnTypeChange", "always")
    .table(silver_layer.bronze_table_name)
    .writeStream.foreachBatch(process_microbatch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

In [0]:
try:
    batch_df = spark.table("workspace.`e-commerce`.sink")

    print("คอลัมน์ใน Bronze")
    print(batch_df.columns)

    bronze_df = silver_layer.read_add_sk_from_bronze_table(batch_df)

    print("ตรวจขั้นเตรียมข้อมูล")
    bronze_df.printSchema()

    print("ตรวจขั้น invalid")
    invalid_df = silver_layer.get_invalid_record(bronze_df)
    invalid_df.printSchema()

except Exception as error:
    print(type(error).__name__)
    print(str(error))